### Metadata
**Name**  
> Naveed Ahmad Hematmal

**Assignment**    

> Multiplayer Perceptrons

# Implementation of Multilayer Perceptrons
:label:`sec_mlp-implementation`

This notebook implements multilayer perceptrons (MLPs) with PyTorch. It first builds a two-layer network by defining its parameters and forward pass, then shows the equivalent concise model with `nn.Sequential`. Both models are trained using a native PyTorch training loop.

In [ ]:
import random

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

random.seed(0)
torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Implementation from Scratch

### Initializing Model Parameters

Fashion-MNIST contains 10 classes, and each grayscale image has $28 \times 28 = 784$ pixels. We flatten each image and build an MLP with one hidden layer of 256 units. Every layer has a weight matrix and bias vector, registered as `nn.Parameter` so PyTorch autograd can compute their gradients.

The model below inherits directly from `torch.nn.Module`. Registering tensors as `nn.Parameter` makes them visible through `model.parameters()` and eligible for optimization.

In [ ]:
class MLPScratch(nn.Module):
    def __init__(self, num_inputs=784, num_outputs=10, num_hiddens=256, sigma=0.01):
        super().__init__()
        self.num_inputs = num_inputs
        self.W1 = nn.Parameter(torch.randn(num_inputs, num_hiddens) * sigma)
        self.b1 = nn.Parameter(torch.zeros(num_hiddens))
        self.W2 = nn.Parameter(torch.randn(num_hiddens, num_outputs) * sigma)
        self.b2 = nn.Parameter(torch.zeros(num_outputs))

    def forward(self, X):
        X = X.reshape(X.shape[0], self.num_inputs)
        hidden = relu(X @ self.W1 + self.b1)
        return hidden @ self.W2 + self.b2

### Model

To make the computation explicit, we define the ReLU activation as $\mathrm{ReLU}(x) = \max(0, x)$.

In [ ]:
def relu(X):
    return torch.clamp_min(X, 0)

Instantiate the model and check the output shape. The output contains one unnormalized score (logit) per class.

In [ ]:
scratch_model = MLPScratch().to(device)
sample_batch = torch.rand(4, 1, 28, 28, device=device)
scratch_model(sample_batch).shape

### Training

Load Fashion-MNIST directly with `torchvision` and use PyTorch `DataLoader` objects for minibatching. The training function below uses `CrossEntropyLoss`, autograd, and SGD, and reports the loss and accuracy on both splits after every epoch.

In [ ]:
batch_size = 256
num_epochs = 10
learning_rate = 0.1

transform = transforms.ToTensor()
train_dataset = datasets.FashionMNIST(root="../data", train=True, transform=transform, download=True)
test_dataset = datasets.FashionMNIST(root="../data", train=False, transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)


def evaluate(model, data_loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    with torch.no_grad():
        for images, targets in data_loader:
            images, targets = images.to(device), targets.to(device)
            logits = model(images)
            total_loss += F.cross_entropy(logits, targets, reduction="sum").item()
            total_correct += (logits.argmax(dim=1) == targets).sum().item()
            total_examples += targets.shape[0]
    return total_loss / total_examples, total_correct / total_examples


def train_model(model, train_loader, test_loader, epochs=10, lr=0.1):
    model = model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = {"train_loss": [], "train_accuracy": [], "test_loss": [], "test_accuracy": []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_examples = 0
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = F.cross_entropy(logits, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * targets.shape[0]
            total_correct += (logits.argmax(dim=1) == targets).sum().item()
            total_examples += targets.shape[0]

        train_loss = total_loss / total_examples
        train_accuracy = total_correct / total_examples
        test_loss, test_accuracy = evaluate(model, test_loader)
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["test_loss"].append(test_loss)
        history["test_accuracy"].append(test_accuracy)
        print(
            f"Epoch {epoch + 1:2d}/{epochs} | "
            f"train loss {train_loss:.4f}, accuracy {train_accuracy:.2%} | "
            f"test loss {test_loss:.4f}, accuracy {test_accuracy:.2%}"
        )
    return history


scratch_model = MLPScratch(num_hiddens=256)
scratch_history = train_model(
    scratch_model, train_loader, test_loader, epochs=num_epochs, lr=learning_rate
)

## Concise Implementation

The same network can be expressed with built-in PyTorch layers. `Flatten` preserves the batch dimension, and the fully connected layers and ReLU perform the same transformations as `MLPScratch`.

In [ ]:
class MLP(nn.Module):
    def __init__(self, num_outputs=10, num_hiddens=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, num_hiddens),
            nn.ReLU(),
            nn.Linear(num_hiddens, num_outputs),
        )

    def forward(self, X):
        return self.net(X)

`nn.Sequential` applies each layer in order, so the concise model needs only to define its sequence. We train it with the same function and data loaders as the scratch implementation, making the architectures directly comparable.

In [ ]:
concise_model = MLP(num_outputs=10, num_hiddens=256)
concise_history = train_model(
    concise_model, train_loader, test_loader, epochs=num_epochs, lr=learning_rate
)

## Summary

Both implementations represent the same two-layer MLP. The scratch model exposes parameter tensors and the forward computation; the concise model uses `nn.Linear`, `nn.ReLU`, and `nn.Sequential`. Both use the same PyTorch data loaders, cross-entropy objective, autograd, SGD optimizer, and evaluation loop.

## Exercises

1. Change `num_hiddens` and compare the test accuracy of the scratch and concise models.
2. Add another hidden layer to each implementation and compare the results.
<!-- 1. Why might a hidden layer with a single neuron limit what the network can learn? -->
3. Change the learning rate and number of epochs. Plot or compare the losses and accuracies in `scratch_history` and `concise_history`.
4. Compare other activation functions and weight initializations.
<!-- 1. Compare training speed for the two implementations on CPU and, when available, GPU.

**Setup:** Requires PyTorch, torchvision, and matplotlib. Fashion-MNIST is downloaded into `../data` the first time the dataset cell runs. -->